In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(".../data/aved_raw.csv")
print(f"Original rows: {len(df)}")

time_col = 'time' 
if time_col not in df.columns:
    time_col = df.columns[0]
    print(f"Using '{time_col}' as time column.")

df[time_col] = pd.to_datetime(df[time_col], utc=True)
df.set_index(time_col, inplace=True)

orig_tz = df.index.tz
df.index = df.index.tz_localize(None)

df.index = df.index.floor('1T')

new_times = df.index.values.copy()
minutes = df.index.minute
even_mask = (minutes % 2 == 0)
new_times[even_mask] -= np.timedelta64(1, 'm')
df.index = pd.DatetimeIndex(new_times)

df_aligned = df.groupby(level=0).first()

df_aligned = df_aligned[df_aligned.index.minute % 2 != 0]

if orig_tz is not None:
    df_aligned.index = df_aligned.index.tz_localize(orig_tz)
df_aligned = df_aligned.sort_index()

print(f"--- Completed ---")
print(f"Aligned total row: {len(df_aligned)}")
print(f"Even timestamp exists: {(df_aligned.index.minute % 2 == 0).any()}")

cols_to_show = [c for c in ['BIOLOGY.LINE 3 TANK 1.N2O value', 'BIOLOGY.LINE 3 TANK 1.NH4 value'] if c in df_aligned.columns]
print(df_aligned[cols_to_show].head())

Original rows: 906815


C:\Users\lenovo\AppData\Local\Temp\ipykernel_28432\206085825.py:26: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df.index = df.index.floor('1T')


--- Completed ---
Aligned total row: 526302
Even timestamp exists: False
                           BIOLOGY.LINE 3 TANK 1.N2O value  \
2022-06-11 22:01:00+00:00                         0.700231   
2022-06-11 22:03:00+00:00                         0.697917   
2022-06-11 22:05:00+00:00                         0.656829   
2022-06-11 22:07:00+00:00                         0.593171   
2022-06-11 22:09:00+00:00                         0.584491   

                           BIOLOGY.LINE 3 TANK 1.NH4 value  
2022-06-11 22:01:00+00:00                          1.66608  
2022-06-11 22:03:00+00:00                          1.52727  
2022-06-11 22:05:00+00:00                          1.43633  
2022-06-11 22:07:00+00:00                          1.39241  
2022-06-11 22:09:00+00:00                          1.40954  
